In [ ]:
!pip install -q groq sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 29.6 MB/s eta 0:00:00


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import faiss

from getpass import getpass
from groq import Groq
from sentence_transformers import SentenceTransformer

In [ ]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

MODEL = "openai/gpt-oss-120b"

print("Groq client ready.")

In [ ]:
DATA_PATH = "/content/spotify_customer_classifiedtwice.csv"

df = pd.read_csv(DATA_PATH)

df = df.dropna(
    subset=["customer_tweet", "spotify_reply"]
).reset_index(drop=True)

df["customer_tweet"] = df["customer_tweet"].astype(str)
df["spotify_reply"] = df["spotify_reply"].astype(str)

print("Historical cases:", len(df))
print(df.columns.tolist())

display(df.head())

Historical cases: 14639
['customer_tweet', 'spotify_reply', 'intent']


,customer_tweet,spotify_reply,intent
0,doesn t work and i even tried deleting the app,hmm can you try restarting your device by hold...,app_technical_issue
1,premium amp when i have it on shuffle it turns...,thanks just to be sure are you free or premium...,playback_issue
2,iphone and i have the most recent update for s...,hey what device operating system and spotify v...,app_technical_issue
3,yes multiple times no changes i am premium use...,got it it s not possible at the moment but we ...,premium_subscription
4,and there is no way to manage albums recently ...,hey there that doesn t sound good what s happe...,app_technical_issue


In [ ]:
INTENT_DEFINITIONS = {

    "playback_issue":
        "Trouble playing, starting, pausing, skipping, buffering, or stopping music.",

    "login_account_issue":
        "Trouble logging in, accessing an account, password, or account access.",

    "premium_subscription":
        "Questions or problems about Premium plans, subscribing, upgrading, or Premium features.",

    "billing_payment":
        "Problems involving charges, payments, billing, invoices, duplicate charges, or payment methods.",

    "playlist_issue":
        "Problems creating, editing, saving, finding, or managing playlists.",

    "app_technical_issue":
        "Problems with the Spotify app such as crashes, freezing, bugs, installation, or technical errors.",

    "missing_unavailable_content":
        "Songs, albums, podcasts, artists, or other Spotify content that is missing or unavailable.",

    "audio_quality_issue":
        "Problems involving sound quality, distorted audio, volume, or audio quality settings.",

    "cancellation_request":
        "The customer is asking about or requesting cancellation of a Premium subscription.",

    "feature_question":
        "The customer is asking how a Spotify feature works or asking about a feature.",

    "complaint":
        "General dissatisfaction where no more specific primary support intent is clear.",

    "other":
        "Casual conversation, appreciation, greetings, unrelated messages, or messages that do not clearly fit another intent."
}

In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Creating historical embeddings...")

historical_embeddings = embedding_model.encode(
    df["customer_tweet"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

historical_embeddings = historical_embeddings.astype("float32")

dimension = historical_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(historical_embeddings)

print("FAISS index created.")
print("Embedding dimension:", dimension)
print("Historical cases indexed:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Creating historical embeddings...


Batches:   0%|          | 0/458 [00:00<?, ?it/s]

FAISS index created.
Embedding dimension: 384
Historical cases indexed: 14639


In [ ]:
def retrieve_similar_cases(customer_message, top_k=3):

    query_embedding = embedding_model.encode(
        [customer_message],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    similarities, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for similarity, idx in zip(
        similarities[0],
        indices[0]
    ):

        # Safety check
        if idx < 0 or idx >= len(df):
            continue

        row = df.iloc[idx]

        case = {
            "customer_tweet": str(row["customer_tweet"]),
            "spotify_reply": str(row["spotify_reply"]),
            "similarity": float(similarity)
        }

        # Historical intent is optional context only.
        # Remember: your current intent column is provisional.
        if "intent" in df.columns:
            case["historical_intent"] = str(row["intent"])

        results.append(case)

    return results

In [ ]:
test_message = "my spotify music keeps stopping"

cases = retrieve_similar_cases(
    test_message,
    top_k=3
)

for i, case in enumerate(cases, 1):

    print("\n" + "=" * 70)
    print(f"HISTORICAL CASE {i}")
    print("=" * 70)

    print("Customer:")
    print(case["customer_tweet"])

    print("\nSpotify Reply:")
    print(case["spotify_reply"])

    if "historical_intent" in case:
        print("\nHistorical Intent:")
        print(case["historical_intent"])

    print("\nSimilarity:")
    print(round(case["similarity"], 3))


HISTORICAL CASE 1
Customer:
my spotify premium thing just stopped working

Spotify Reply:
hi there how can we help can you tell us more about what s going on we ll see what we can suggest nh

Historical Intent:
playback_issue

Similarity:
0.787

HISTORICAL CASE 2
Customer:
so this problem has continually happened for the past days and i m a little sick of restarting my phone in order for spotify to work any more suggestions

Spotify Reply:
great to hear it s now working if the problem reoccur give us a shout and we ll come running pl

Historical Intent:
app_technical_issue

Similarity:
0.784

HISTORICAL CASE 3
Customer:
i deleted spotify and downloaded it again but it s still doing the same thing

Spotify Reply:
hey there don t worry we can help what s happening exactly can you let us know the device os you re using we ll see what we can suggest ji

Historical Intent:
app_technical_issue

Similarity:
0.782


In [ ]:
def analyze_and_generate_reply(customer_message, historical_cases):

    intent_text = "\n".join(
        [
            f"- {intent}: {definition}"
            for intent, definition in INTENT_DEFINITIONS.items()
        ]
    )

    historical_text = ""

    for i, case in enumerate(historical_cases, 1):

        historical_text += f"""
HISTORICAL CASE {i}

Customer:
{case["customer_tweet"]}

Spotify Reply:
{case["spotify_reply"]}

Similarity:
{case["similarity"]:.3f}

"""

    prompt = f"""
You are an AI customer-support agent for SpotifyCares.

Your job is to understand the customer's message, use the
retrieved historical Spotify support cases as references when
useful, and decide whether the case should be handled by an
AI AGENT or HUMAN AGENT.

CUSTOMER MESSAGE:
{customer_message}

HISTORICAL CASES:
{historical_text}

AVAILABLE INTENTS:

{intent_text}


========================
INTENT DECISION
========================

Choose exactly ONE primary intent.

Understand the entire customer message.

Do not classify based only on keywords.

If cancellation is mentioned while the customer is actually
describing another problem, classify the underlying problem.

For example:

"I will cancel if you cannot fix my playback problem"

should be playback_issue if playback is the actual problem.

Use cancellation_request when cancellation itself is the
actual request.

Use other for casual conversation, greetings, appreciation,
or messages that are not a specific support issue.


========================
HANDLING DECISION
========================

Choose:

AI AGENT

OR

HUMAN AGENT

The intent does NOT automatically determine the handling.

Any intent can be handled by either AI AGENT or HUMAN AGENT.

Use AI AGENT when:
- the problem is clear
- useful guidance can be provided
- historical cases provide useful information
- no account-specific investigation is required
- the customer is asking a normal informational question
- the message is casual or conversational

Use HUMAN AGENT when:
- the customer explicitly asks for a human
- previous troubleshooting has failed
- account-specific investigation is required
- the AI cannot safely resolve the issue
- manual action is required
- the situation is unusual or requires additional review

Do not choose HUMAN AGENT simply because:
- the intent is other
- cancellation is mentioned
- billing is mentioned
- an account is mentioned


========================
REPLY
========================

If HANDLED_BY is AI AGENT:

Generate a helpful customer-facing response.

Use the historical Spotify responses as references when
they are relevant.

Do not blindly copy historical responses.

Use reasonable common-sense reasoning to adapt the response.

Do not invent:
- refunds
- account changes
- policies
- prices
- guarantees
- actions that were not performed


If HANDLED_BY is HUMAN AGENT:

Do NOT attempt to solve the issue.

Instead, politely tell the customer that additional assistance
is required and direct them to the Spotify customer-care number.

The customer-care number will be supplied separately by the
system.


OUTPUT EXACTLY:

INTENT: <intent>
HANDLED_BY: <AI AGENT or HUMAN AGENT>
REASON: <short reason>
REPLY: <customer-facing response>
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a careful Spotify customer-support agent."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=350
    )

    text = response.choices[0].message.content.strip()

    print("\nDEBUG LLM OUTPUT:")
    print(text)

    # -----------------------------
    # Parse intent
    # -----------------------------

    intent_match = re.search(
        r"INTENT\s*:\s*([a-zA-Z0-9_]+)",
        text,
        re.IGNORECASE
    )

    intent = "other"

    if intent_match:
        candidate = intent_match.group(1).lower().strip()

        if candidate in INTENT_DEFINITIONS:
            intent = candidate

    # -----------------------------
    # Parse handling
    # -----------------------------

    handling_match = re.search(
        r"HANDLED_BY\s*:\s*(AI\s*AGENT|HUMAN\s*AGENT)",
        text,
        re.IGNORECASE
    )

    if handling_match:

        handled_by = re.sub(
            r"\s+",
            " ",
            handling_match.group(1).upper()
        ).strip()

    else:

        handled_by = "HUMAN AGENT"

    # -----------------------------
    # Parse reason
    # -----------------------------

    reason_match = re.search(
        r"REASON\s*:\s*(.*?)(?=\n\s*REPLY\s*:|\Z)",
        text,
        re.IGNORECASE | re.DOTALL
    )

    if reason_match:
        reason = reason_match.group(1).strip()
    else:
        reason = "Additional review is required."

    # -----------------------------
    # Parse reply
    # -----------------------------

    reply_match = re.search(
        r"REPLY\s*:\s*(.*)",
        text,
        re.IGNORECASE | re.DOTALL
    )

    if reply_match:
        reply = reply_match.group(1).strip()
    else:
        reply = ""

    # --------------------------------------------------
    # HUMAN AGENT OVERRIDE
    # --------------------------------------------------
    # The customer-care number is controlled by our system,
    # NOT generated by the LLM.
    # --------------------------------------------------
    CUSTOMER_CARE_NUMBER="6303815555"
    if handled_by == "HUMAN AGENT":

        reply = (
            "Thanks for reaching out. This issue requires "
            "additional assistance from our customer support team. "
            f"Please contact Spotify Customer Care at "
            f"{CUSTOMER_CARE_NUMBER} for further help."
        )

    return {
        "intent": intent,
        "handled_by": handled_by,
        "reason": reason,
        "reply": reply
    }

In [ ]:
def support_agent(customer_message, top_k=3):

    # ---------------------------------------------------------
    # STEP 1
    # Retrieve historical Spotify cases
    # ---------------------------------------------------------

    historical_cases = retrieve_similar_cases(
        customer_message,
        top_k=top_k
    )

    # ---------------------------------------------------------
    # STEP 2
    # Give customer + retrieved evidence to LLM
    # ---------------------------------------------------------

    analysis = analyze_and_generate_reply(
        customer_message,
        historical_cases
    )

    # ---------------------------------------------------------
    # STEP 3
    # Return everything
    # ---------------------------------------------------------

    return {
        "customer_tweet": customer_message,
        "intent": analysis["intent"],
        "handled_by": analysis["handled_by"],
        "reason": analysis["reason"],
        "reply": analysis["reply"],
        "historical_cases": historical_cases
    }

In [ ]:
def print_agent_result(result):

    print("\n" + "=" * 70)
    print("SPOTIFY CUSTOMER SUPPORT AGENT")
    print("=" * 70)

    print("\nCustomer Tweet:\n")
    print(result["customer_tweet"])

    print("\nPredicted Intent:")
    print(result["intent"])

    print("\nHandled By:")
    print(result["handled_by"])

    print("\nReason:")
    print(result["reason"])

    print("\nSpotify Reply:\n")
    print(result["reply"])

    print("\n" + "=" * 70)
    print("TOP 3 HISTORICAL CASES")
    print("=" * 70)

    for i, case in enumerate(
        result["historical_cases"],
        1
    ):

        print(f"\nCASE {i}")
        print("-" * 50)

        print("Customer:")
        print(case["customer_tweet"])

        print("\nSpotify Reply:")
        print(case["spotify_reply"])

        if "historical_intent" in case:
            print("\nHistorical Intent:")
            print(case["historical_intent"])

        print("\nSimilarity:")
        print(round(case["similarity"], 3))

In [ ]:
customer_message = """
i have a problem with my spotify music ci want to talk to my spotify support team
"""

result = support_agent(
    customer_message,
    top_k=3
)

print_agent_result(result)


DEBUG LLM OUTPUT:


SPOTIFY CUSTOMER SUPPORT AGENT

Customer Tweet:


i have a problem with my spotify music ci want to talk to my spotify support team


Predicted Intent:
other

Handled By:
HUMAN AGENT

Reason:
Additional review is required.

Spotify Reply:

Thanks for reaching out. This issue requires additional assistance from our customer support team. Please contact Spotify Customer Care at 6303815555 for further help.

TOP 3 HISTORICAL CASES

CASE 1
--------------------------------------------------
Customer:
i ve tried to contact spotify every way possible

Spotify Reply:
just to confirm did you request access to spotify for artists already also we suggest you delete your last tweet since it s public bh

Historical Intent:
other

Similarity:
0.773

CASE 2
--------------------------------------------------
Customer:
hey thanks for the reply but this problem is just happening on spotify should i still reach out to them

Spotify Reply:
hey there you ll need to reach out to your di